In [1]:
# imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from PIL import Image
import os

import time
print("Finished", time.localtime())

Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=14, tm_min=22, tm_sec=5, tm_wday=2, tm_yday=120, tm_isdst=1)


# Load Datasets into Dataframe

In [2]:
# Path to your FER2013 CSV file
CSV_PATH = 'fer2013.csv'

# Load CSV
df = pd.read_csv(CSV_PATH)

# Only use "Training" and "PublicTest" for simplicity
df = df[df['Usage'].isin(['Training', 'PublicTest'])]

# Map emotion: 3 (happy) -> 1, others -> 0
df['label'] = (df['emotion'] == 3).astype(int)

# Undersample "not happy" to match "happy"
happy = df[df['label'] == 1]
not_happy = df[df['label'] == 0].sample(len(happy), random_state=42)
df_balanced = pd.concat([happy, not_happy]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Happy: {len(happy)}, Not Happy (sampled): {len(not_happy)}")
print(df_balanced['label'].value_counts())

print("Finished", time.localtime())

Happy: 8110, Not Happy (sampled): 8110
label
0    8110
1    8110
Name: count, dtype: int64
Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=14, tm_min=22, tm_sec=7, tm_wday=2, tm_yday=120, tm_isdst=1)


In [3]:
class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pixels = np.fromstring(row['pixels'], sep=' ', dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(pixels).convert('RGB')
        label = row['label']
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms for EfficientNetB0 (224x224)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Finished", time.localtime())


Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=14, tm_min=22, tm_sec=7, tm_wday=2, tm_yday=120, tm_isdst=1)


# Model Prep

In [4]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def get_model():
    weights = EfficientNet_B0_Weights.DEFAULT  # oder IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 2)  # Binary classification
    return model

print("Finished", time.localtime())

Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=14, tm_min=22, tm_sec=7, tm_wday=2, tm_yday=120, tm_isdst=1)


# Training

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
BATCH_SIZE = 16
EPOCHS = 10
K_FOLDS = 5

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
X = df_balanced.index.values
y = df_balanced['label'].values

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{K_FOLDS} ---")
    train_df = df_balanced.iloc[train_idx]
    val_df = df_balanced.iloc[val_idx]

    train_ds = FERDataset(train_df, transform=transform)
    val_ds = FERDataset(val_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = get_model().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader.dataset):.4f}")

    # Validation
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.numpy())
    acc = accuracy_score(targets, preds)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")
    print(classification_report(targets, preds, target_names=['Not Happy', 'Happy']))
    fold_results.append((acc, classification_report(targets, preds, target_names=['Not Happy', 'Happy'], output_dict=True)))

print("Finished", time.localtime())

cuda

--- Fold 1/5 ---


/home/marvin/Developer/NoSmiles/local/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:310.)
  return F.linear(input, self.weight, self.bias)


Epoch 1/10, Loss: 0.3692
Epoch 2/10, Loss: 0.2241
Epoch 3/10, Loss: 0.1563
Epoch 4/10, Loss: 0.1014
Epoch 5/10, Loss: 0.0730
Epoch 6/10, Loss: 0.0589
Epoch 7/10, Loss: 0.0453
Epoch 8/10, Loss: 0.0374
Epoch 9/10, Loss: 0.0325
Epoch 10/10, Loss: 0.0317
Fold 1 Accuracy: 0.8998
              precision    recall  f1-score   support

   Not Happy       0.93      0.86      0.90      1622
       Happy       0.87      0.94      0.90      1622

    accuracy                           0.90      3244
   macro avg       0.90      0.90      0.90      3244
weighted avg       0.90      0.90      0.90      3244


--- Fold 2/5 ---
Epoch 1/10, Loss: 0.3687
Epoch 2/10, Loss: 0.2254
Epoch 3/10, Loss: 0.1556
Epoch 4/10, Loss: 0.1077
Epoch 5/10, Loss: 0.0720
Epoch 6/10, Loss: 0.0552
Epoch 7/10, Loss: 0.0462
Epoch 8/10, Loss: 0.0427
Epoch 9/10, Loss: 0.0420
Epoch 10/10, Loss: 0.0259
Fold 2 Accuracy: 0.9078
              precision    recall  f1-score   support

   Not Happy       0.91      0.91      0.91      1

# Evaluation

In [6]:
# Summarize results
accs = [r[0] for r in fold_results]
print(f"\nMean Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")

print("Finished", time.localtime())


Mean Accuracy: 0.9065 ± 0.0034
Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=15, tm_min=36, tm_sec=16, tm_wday=2, tm_yday=120, tm_isdst=1)


# Saving

In [7]:
final_model = get_model().to(device)
final_ds = FERDataset(df_balanced, transform=transform)
final_loader = DataLoader(final_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    final_model.train()
    for imgs, labels in final_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Save model
os.makedirs('models', exist_ok=True)
torch.save(final_model.state_dict(), 'models/efficientnetb0_fer2013_happy.pth')
print("Model saved to models/efficientnetb0_fer2013_happy.pth")

print("Finished", time.localtime())

Model saved to models/efficientnetb0_fer2013_happy.pth
Finished time.struct_time(tm_year=2025, tm_mon=4, tm_mday=30, tm_hour=15, tm_min=54, tm_sec=19, tm_wday=2, tm_yday=120, tm_isdst=1)
